<a href="https://colab.research.google.com/github/dsp130806/Hollywood-Movie-Recommendation-System/blob/main/Tollywood_Movie_Recommendation_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import pandas as pd

telugu_movies = pd.read_csv('TeluguMovies_dataset.csv')

print("Shape:", telugu_movies.shape)
print("Columns:", telugu_movies.columns.tolist())

telugu_movies.head()

Shape: (1400, 9)
Columns: ['Unnamed: 0', 'Movie', 'Year', 'Certificate', 'Genre', 'Overview', 'Runtime', 'Rating', 'No.of.Ratings']


,Unnamed: 0,Movie,Year,Certificate,Genre,Overview,Runtime,Rating,No.of.Ratings
0,0,Bahubali: The Beginning,2015.0,UA,"Action, Drama","In ancient India, an adventurous and darin...",159,8.1,99114
1,1,Baahubali 2: The Conclusion,2017.0,UA,"Action, Drama","When Shiva, the son of Bahubali, learns ab...",167,8.2,71458
2,2,1 - Nenokkadine,2014.0,UA,"Action, Thriller",A rock star must overcome his psychologica...,170,8.1,42372
3,3,Dhoom:3,2013.0,UA,"Action, Thriller","When Sahir, a circus entertainer trained i...",172,5.4,42112
4,4,Ra.One,2011.0,U,"Action, Adventure, Sci-Fi",When the titular antagonist of an action g...,156,4.6,37211


In [4]:
print(telugu_movies.columns.tolist())


['Unnamed: 0', 'Movie', 'Year', 'Certificate', 'Genre', 'Overview', 'Runtime', 'Rating', 'No.of.Ratings']


In [5]:
telugu_movies = telugu_movies.drop(columns=['Unnamed: 0'])

print(telugu_movies.isnull().sum())

Movie              0
Year              48
Certificate      449
Genre             11
Overview         179
Runtime            0
Rating             0
No.of.Ratings      0
dtype: int64


In [6]:
telugu_movies['Genre'] = telugu_movies['Genre'].fillna('')
telugu_movies['Overview'] = telugu_movies['Overview'].fillna('')

print("Rows remaining:", telugu_movies.shape[0])

Rows remaining: 1400


In [7]:
telugu_movies['Genre'] = telugu_movies['Genre'].fillna('')
telugu_movies['Overview'] = telugu_movies['Overview'].fillna('')

print("Rows remaining:", telugu_movies.shape[0])

Rows remaining: 1400


In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

vectorizer = TfidfVectorizer(max_features=5000, stop_words='english')
vectors = vectorizer.fit_transform(telugu_movies['tags']).toarray()

similarity = cosine_similarity(vectors)

print("Vectors shape:", vectors.shape)
print("Similarity shape:", similarity.shape)

KeyError: 'tags'

In [9]:
print(telugu_movies.columns.tolist())


['Movie', 'Year', 'Certificate', 'Genre', 'Overview', 'Runtime', 'Rating', 'No.of.Ratings']


In [10]:
telugu_movies['tags'] = telugu_movies['Genre'] + ' ' + telugu_movies['Overview']
telugu_movies['tags'] = telugu_movies['tags'].apply(lambda x: x.lower())

telugu_movies[['Movie', 'tags']].head()

,Movie,tags
0,Bahubali: The Beginning,"action, drama in ancient india..."
1,Baahubali 2: The Conclusion,"action, drama when shiva, the ..."
2,1 - Nenokkadine,"action, thriller a rock star m..."
3,Dhoom:3,"action, thriller when sahir, a..."
4,Ra.One,"action, adventure, sci-fi when..."


In [11]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

vectorizer = TfidfVectorizer(max_features=5000, stop_words='english')
vectors = vectorizer.fit_transform(telugu_movies['tags']).toarray()

similarity = cosine_similarity(vectors)

print("Vectors shape:", vectors.shape)
print("Similarity shape:", similarity.shape)

Vectors shape: (1400, 5000)
Similarity shape: (1400, 1400)


In [13]:
def recommend_telugu(movie_name):
    movie_name = movie_name.lower()
    matches = telugu_movies[telugu_movies['Movie'].str.lower() == movie_name]

    if matches.empty:
        return "Movie not found in database. Check spelling!"

    idx = matches.index[0]

    distances = similarity[idx]

    movie_list = sorted(list(enumerate(distances)), reverse=True, key=lambda x: x[1])[1:6]

    recommendations = []
    for i in movie_list:
        recommendations.append(telugu_movies.iloc[i[0]]['Movie'])

    return recommendations

# Try it!
print(recommend_telugu("Bahubali: The Beginning"))

['Sree', 'Assembly Rowdy', 'Chinnadana Nee Kosam', 'Rowdy Fellow', 'Oh My Friend']


In [15]:
print(recommend_telugu("Arjun Reddy"))
print(recommend_telugu("Mansara"))

['Samarasimha Reddy', 'Naa Peru Surya Na Illu India', 'Coolie No. 1', 'Sitaramaraju', 'Two Town Rowdy']
['Undiporaadhey', 'Prema Pipasi', 'Bommarillu', 'Amrutha Ramam', 'Tholi Prema']


In [16]:
!pip install gradio -q

import gradio as gr

def recommend_ui(movie_name):
    result = recommend_telugu(movie_name)
    if isinstance(result, str):
        return result
    return "\n".join(result)

interface = gr.Interface(
    fn=recommend_ui,
    inputs=gr.Textbox(placeholder="Enter a Telugu movie name, e.g. Baahubali: The Beginning"),
    outputs="text",
    title="Tollywood Movie Recommendation System",
    description="Enter a Telugu movie you like, and get 5 similar movie suggestions!"
)

interface.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://f4e065112ba5994768.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
